# 02 — Chọn mẫu và kiểm tra split

Mỗi dòng là một bản ghi review. Chọn bản ghi có giá quan sát và nhãn `points`; giữ giá thiếu trong cleaned data để audit. Không fit preprocessing ở đây.

Bộ chia theo `title` chuẩn hóa, seed 42, đã chốt ngày 21/10/2026. Notebook chỉ tái tạo trong bộ nhớ và kiểm tra với CSV đã có, không ghi đè chúng. EDA trước đây đã xem toàn mẫu và holdout 42 từng được thử; không khẳng định test chưa từng được quan sát.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.split_data import create_splits, check_existing_splits, review_overlap
cleaned = pd.read_csv(ROOT / 'data/processed/wine_cleaned_final.csv')
display(pd.DataFrame({'sample': ['cleaned', 'observed_price', 'missing_price'],
                      'rows': [len(cleaned), cleaned.price.notna().sum(), cleaned.price.isna().sum()]}))
parts = create_splits(cleaned)
check_existing_splits(parts)
display(pd.DataFrame([{'split': name, 'rows': len(part), 'title_groups': part.title_clean.nunique(),
                       'missing_price': part.price.isna().sum(), 'missing_points': part.points.isna().sum()}
                      for name, part in parts.items()]))


,sample,rows
0,cleaned,119988
1,observed_price,111593
2,missing_price,8395


,split,rows,title_groups,missing_price,missing_points
0,train,78106,77437,0,0
1,val,16745,16594,0,0
2,test,16742,16594,0,0


## Chống chồng nhóm và kiểm tra văn bản

Kiểm tra giao nhóm tên rượu bằng 0. Báo cáo văn bản chỉ so chuỗi review sau chuyển chữ thường và gộp khoảng trắng; không dùng nhãn validation/test. Số trùng là số chuỗi review phân biệt dùng chung, không phải số dòng bị trùng. Không tự chia lại dữ liệu.

In [2]:
from itertools import combinations
for left, right in combinations(parts, 2):
    assert not (set(parts[left].title_clean) & set(parts[right].title_clean))
overlap = review_overlap(parts)
display(overlap)
shared = int(overlap.shared_unique_reviews.sum())
display(Markdown(f"**Nhận xét:** tổng số giao review phân biệt theo từng cặp tập là {shared:,}. "
    "Nếu có giao, đây là giới hạn của đánh giá văn bản cần xử lý trước thí nghiệm chính; "
    "không thay split để chọn kết quả thuận lợi. Các nhóm title không giao nhau."))

,left,right,shared_unique_reviews
0,train,val,6
1,train,test,5
2,val,test,1


**Nhận xét:** tổng số giao review phân biệt theo từng cặp tập là 12. Nếu có giao, đây là giới hạn của đánh giá văn bản cần xử lý trước thí nghiệm chính; không thay split để chọn kết quả thuận lợi. Các nhóm title không giao nhau.

## Kết luận

Module dùng chung tái tạo chính xác thứ tự bản ghi và nhóm của các file đã lưu. Giá thiếu không được điền trước split; niên vụ thiếu được giữ để xử lý bằng bộ điền thiếu fit trên train ở giai đoạn mô hình. Không tính metric, không xem phân phối nhãn test để điều chỉnh phương pháp.